<a href="https://colab.research.google.com/github/Somrat390/NLP/blob/main/Bag_of_word.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from sklearn.feature_extraction.text import CountVectorizer

v = CountVectorizer()
v.fit(['Thor eat pizza'])
v.vocabulary_

{'thor': 2, 'eat': 0, 'pizza': 1}

In [4]:
v1 = CountVectorizer(ngram_range=(1,2))
v1.fit(['Thor eat pizza'])
v1.vocabulary_

{'thor': 3, 'eat': 0, 'pizza': 2, 'thor eat': 4, 'eat pizza': 1}

In [5]:
!pip -q install spacy

In [6]:
import spacy

nlp  = spacy.load('en_core_web_sm')


def preprocessor(text):
  doc = nlp(text)

  filterd_content = []

  for token in doc:
    if token.is_stop or token.is_punct:
      continue
    filterd_content.append(token.lemma_)

  return " ".join(filterd_content)


In [7]:
preprocessor("Lokking eating piaaza djfeijt")

'lokke eat piaaza djfeijt'

In [8]:
corpus = [
    "Thor ate pizza",
    "Loki is tall",
    "Loki is eating pizza"
]

corpus_processed = [preprocessor(text) for text in corpus]
corpus_processed

['thor eat pizza', 'Loki tall', 'Loki eat pizza']

In [9]:
v = CountVectorizer(ngram_range=(1,2))
v.fit(corpus_processed)
v.vocabulary_

{'thor': 7,
 'eat': 0,
 'pizza': 5,
 'thor eat': 8,
 'eat pizza': 1,
 'loki': 2,
 'tall': 6,
 'loki tall': 4,
 'loki eat': 3}

In [10]:
v.transform(["Thor eat pizza"]).toarray()

array([[1, 1, 0, 0, 0, 1, 0, 1, 1]])

In [11]:
import pandas as pd

df = pd.read_json("news_dataset.json")

In [13]:
print(df.shape)

(12695, 2)


In [14]:
df.head()

,text,category
0,Watching Schrödinger's Cat Die University of C...,SCIENCE
1,WATCH: Freaky Vortex Opens Up In Flooded Lake,SCIENCE
2,Entrepreneurs Today Don't Need a Big Budget to...,BUSINESS
3,These Roads Could Recharge Your Electric Car A...,BUSINESS
4,Civilian 'Guard' Fires Gun While 'Protecting' ...,CRIME


In [16]:
df.category.value_counts()

,count
category,
BUSINESS,4254
SPORTS,4167
CRIME,2893
SCIENCE,1381


In [17]:
min_sample = 1381
df_business = df[df.category=="BUSINESS"].sample(min_sample, random_state=2022)
df_sports = df[df.category=="SPORTS"].sample(min_sample, random_state=2022)
df_crime = df[df.category=="CRIME"].sample(min_sample, random_state=2022)
df_science = df[df.category=="SCIENCE"].sample(min_sample, random_state=2022)


In [18]:
df_balanced = pd.concat([df_business,df_sports,df_crime,df_science], axis=0)
df_balanced.category.value_counts()

,count
category,
BUSINESS,1381
SPORTS,1381
CRIME,1381
SCIENCE,1381


In [19]:
df_balanced['category_num'] = df_balanced.category.map(
    {
        'BUSINESS': 0,
        'SPORTS': 1,
        'CRIME': 2,
        'SCIENCE': 3
    }
)

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df_balanced.text,
    df_balanced.category_num,
    test_size=0.2,
    random_state = 22,
    stratify = df_balanced.category_num
)

In [22]:
print(X_train.shape)
X_train.head()

(4419,)


,text
1291,He Saw Things No One Else Would For Another 50...
1329,Scientists Lobbying To Restore Pluto As A Plan...
6589,'Bowie' The Baby Penguin Is Adorable Tribute T...
1079,4 Signs You're a Wannabe Business Owner
7375,Amazon May Refund Your Potentially Explosive H...


In [23]:
y_train.value_counts()

,count
category_num,
3,1105
0,1105
2,1105
1,1104


In [24]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

clf = Pipeline([
    ('Vectrorize_bow', CountVectorizer()),
    ('Multi NB', MultinomialNB())
]
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.78      0.91      0.84       276
           1       0.92      0.82      0.87       277
           2       0.87      0.90      0.88       276
           3       0.94      0.85      0.89       276

    accuracy                           0.87      1105
   macro avg       0.88      0.87      0.87      1105
weighted avg       0.88      0.87      0.87      1105



In [25]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

clf = Pipeline([
    ('Vectrorize_bow', CountVectorizer(ngram_range=(1,2))),
    ('Multi NB', MultinomialNB())
]
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.71      0.93      0.81       276
           1       0.93      0.79      0.86       277
           2       0.89      0.88      0.89       276
           3       0.94      0.80      0.87       276

    accuracy                           0.85      1105
   macro avg       0.87      0.85      0.85      1105
weighted avg       0.87      0.85      0.85      1105



In [26]:
df_balanced['preprocessed_text'] = df_balanced.text.apply(preprocessor)

In [27]:
df_balanced.head()

,text,category,category_num,preprocessed_text
11967,GCC Business Leaders Remain Confident in the F...,BUSINESS,0,GCC Business leader remain confident Face Regi...
2912,From the Other Side; an Honest Review from Emp...,BUSINESS,0,Honest Review Employees wake morning love impo...
3408,"Mike McDerment, CEO of FreshBooks, Talks About...",BUSINESS,0,Mike McDerment CEO FreshBooks Talks give build...
502,How to Market Your Business While Traveling th...,BUSINESS,0,market business travel World recently amazing ...
5279,How to Leverage Intuition in Decision-making I...,BUSINESS,0,leverage intuition decision making feel safe r...


In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    df_balanced.preprocessed_text,
    df_balanced.category_num,
    test_size=0.2,
    random_state = 22,
    stratify = df_balanced.category_num
)

In [29]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

clf = Pipeline([
    ('Vectrorize_bow', CountVectorizer(ngram_range=(1,2))),
    ('Multi NB', MultinomialNB())
]
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.84      0.90      0.87       276
           1       0.93      0.88      0.90       277
           2       0.86      0.91      0.88       276
           3       0.93      0.87      0.90       276

    accuracy                           0.89      1105
   macro avg       0.89      0.89      0.89      1105
weighted avg       0.89      0.89      0.89      1105

